# AI-Assisted Workforce Signals Pipeline

An end-to-end Databricks prototype for turning unstructured employee feedback into reviewable workforce signals.

**Data:** 50 synthetic employee-feedback records  
**Clean analytical sample:** 40 records  
**Core workflow:** Synthetic Data → Bronze → Silver → AI-Assisted Gold → Human Review → Management Brief  
**Tools:** PySpark, Spark SQL, Databricks AI Functions

> **Important:** This project uses synthetic data only. It does not contain company or employee data, and the findings should be interpreted as a workflow demonstration rather than real workforce conclusions.


## 1. Synthetic Data Generation

The notebook starts with 40 valid synthetic employee comments across multiple departments, then intentionally introduces data-quality issues:

- 8 duplicate records with capitalization and whitespace variation
- 2 invalid short/blank records
- inconsistent department labels

This creates a 50-row raw dataset that can be used to demonstrate cleaning, normalization, and deduplication.


In [0]:
import random
from datetime import date, timedelta
import pandas as pd

random.seed(42)

# 40 valid synthetic employee comments
base_reviews = [
    ("Engineering", "Mandatory overtime is burning out the team."),
    ("Engineering", "The new data platform saves me hours every week."),
    ("Engineering", "The workload has been unsustainable since the reorganization."),
    ("Engineering", "My manager supports the team and communicates clearly."),
    ("Engineering", "Technical requirements often change halfway through the project."),
    ("Engineering", "I receive useful feedback during code reviews."),

    ("Sales", "The bonus formula is confusing and not transparent."),
    ("Sales", "Pay has not kept up with the market."),
    ("Sales", "The recognition program is genuinely motivating."),
    ("Sales", "Late-night messages have become part of the culture."),
    ("Sales", "My manager gives clear priorities and removes blockers."),
    ("Sales", "Promotion expectations are not clearly explained."),

    ("Customer Success", "Flexible hours help me balance work and family."),
    ("Customer Success", "My work is rarely recognized."),
    ("Customer Success", "Onboarding documentation could be clearer."),
    ("Customer Success", "There is no clear path to promotion."),
    ("Customer Success", "The team is supportive during difficult customer situations."),
    ("Customer Success", "We do not have enough staff to manage the current workload."),

    ("Product", "The learning stipend helped me develop new skills."),
    ("Product", "Leadership decisions are communicated too late."),
    ("Product", "My manager gives helpful feedback during one-on-one meetings."),
    ("Product", "I received no feedback until the annual review."),
    ("Product", "Cross-functional collaboration has improved this year."),
    ("Product", "Too many meetings leave little time for focused work."),

    ("Operations", "Our tools require too much manual spreadsheet work."),
    ("Operations", "The team environment is fine but not especially engaging."),
    ("Operations", "Staffing shortages have not been addressed."),
    ("Operations", "The remote work policy helps me stay productive."),
    ("Operations", "Processes are documented clearly and consistently."),
    ("Operations", "Small errors often require several layers of approval to fix."),

    ("Data", "Senior analysts provide excellent mentorship."),
    ("Data", "Requirements frequently change during projects."),
    ("Data", "My promotion salary increase was fair."),
    ("Data", "Meetings take up most of the day and analysis happens at night."),
    ("Data", "The team encourages experimentation and learning."),
    ("Data", "Access to important datasets takes too long."),

    ("HR", "Employees are not always informed when policies change."),
    ("HR", "The new onboarding process is much more organized."),
    ("HR", "The HR team is handling too many manual requests."),
    ("HR", "Leadership listens to employee feedback and follows up.")
]

department_variants = {
    "Engineering": ["Engineering", "engineering", "Eng"],
    "Sales": ["Sales", "sales"],
    "Customer Success": [
        "Customer Success",
        "customer success",
        "CustSuccess"
    ],
    "Product": ["Product", "product", "Prod"],
    "Operations": ["Operations", "operations", "Ops"],
    "Data": ["Data", "data", "Data Analytics"],
    "HR": ["HR", "hr", "Human Resources"]
}

rows = []
today = date.today()

# Convert the 40 valid comments into complete records
for index, (department, review) in enumerate(base_reviews, start=1001):
    rows.append({
        "employee_id": f"E{index}",
        "department_raw": random.choice(
            department_variants[department]
        ),
        "tenure_years": round(random.uniform(0.3, 9.0), 1),
        "review_date": (
            today - timedelta(days=random.randint(0, 90))
        ).isoformat(),
        "review_text": review
    })

# Intentionally duplicate 8 records and introduce case/whitespace issues
duplicate_indexes = [0, 3, 7, 11, 15, 19, 25, 31]

for index in duplicate_indexes:
    duplicate_row = rows[index].copy()
    duplicate_row["review_text"] = (
        "  " + duplicate_row["review_text"].upper() + "  "
    )
    rows.append(duplicate_row)

# Add 2 intentionally invalid records
rows.append({
    "employee_id": "E9998",
    "department_raw": "Ops",
    "tenure_years": 2.0,
    "review_date": today.isoformat(),
    "review_text": "   "
})

rows.append({
    "employee_id": "E9999",
    "department_raw": "Data",
    "tenure_years": 1.0,
    "review_date": today.isoformat(),
    "review_text": "n/a"
})

# Convert to a pandas DataFrame
pdf = pd.DataFrame(rows)

print("Raw rows:", len(pdf))
display(pdf)


Raw rows: 50


employee_id,department_raw,tenure_years,review_date,review_text
E1001,Eng,1.3,2026-06-28,Mandatory overtime is burning out the team.
E1002,Engineering,2.2,2026-07-20,The new data platform saves me hours every week.
E1003,Eng,6.7,2026-05-25,The workload has been unsustainable since the reorganization.
E1004,Engineering,5.4,2026-07-29,My manager supports the team and communicates clearly.
E1005,Engineering,1.1,2026-07-04,Technical requirements often change halfway through the project.
E1006,Eng,5.5,2026-05-23,I receive useful feedback during code reviews.
E1007,Sales,6.5,2026-05-05,The bonus formula is confusing and not transparent.
E1008,sales,2.2,2026-05-19,Pay has not kept up with the market.
E1009,sales,7.3,2026-08-02,The recognition program is genuinely motivating.
E1010,Sales,6.4,2026-06-20,Late-night messages have become part of the culture.


## 2. Bronze Layer — Raw Data Ingestion

The raw synthetic dataset is converted to a Spark DataFrame and persisted as a Bronze table. At this stage, the records are intentionally kept close to their original form so data-quality issues remain visible.


In [0]:
bronze = spark.createDataFrame(pdf)

print("Bronze rows:", bronze.count())
bronze.printSchema()

display(bronze.limit(10))

Bronze rows: 50
root
 |-- employee_id: string (nullable = true)
 |-- department_raw: string (nullable = true)
 |-- tenure_years: double (nullable = true)
 |-- review_date: string (nullable = true)
 |-- review_text: string (nullable = true)



employee_id,department_raw,tenure_years,review_date,review_text
E1001,Eng,1.3,2026-06-28,Mandatory overtime is burning out the team.
E1002,Engineering,2.2,2026-07-20,The new data platform saves me hours every week.
E1003,Eng,6.7,2026-05-25,The workload has been unsustainable since the reorganization.
E1004,Engineering,5.4,2026-07-29,My manager supports the team and communicates clearly.
E1005,Engineering,1.1,2026-07-04,Technical requirements often change halfway through the project.
E1006,Eng,5.5,2026-05-23,I receive useful feedback during code reviews.
E1007,Sales,6.5,2026-05-05,The bonus formula is confusing and not transparent.
E1008,sales,2.2,2026-05-19,Pay has not kept up with the market.
E1009,sales,7.3,2026-08-02,The recognition program is genuinely motivating.
E1010,Sales,6.4,2026-06-20,Late-night messages have become part of the culture.


In [0]:
bronze.write.mode("overwrite").saveAsTable("bronze_employee_reviews")

In [0]:
spark.table("bronze_employee_reviews")

DataFrame[employee_id: string, department_raw: string, tenure_years: double, review_date: string, review_text: string]

In [0]:
saved_bronze = spark.table("bronze_employee_reviews")
print(saved_bronze.count())

50


## 3. Data Quality Checks

Before building the Silver layer, the workflow checks department values and identifies records with unusably short review text. Valid reviews are retained for normalization and deduplication.


In [0]:
from pyspark.sql import functions as F

department_check = (
    saved_bronze
    .groupBy("department_raw")
    .count()
    .orderBy("department_raw")
)

display(department_check)

department_raw,count
CustSuccess,3
Customer Success,2
Data,4
Data Analytics,3
Eng,4
Engineering,4
HR,1
Human Resources,2
Operations,3
Ops,4


In [0]:
bad_reviews = saved_bronze.filter(
    F.length(F.trim(F.col("review_text"))) < 15
)

display(bad_reviews)

employee_id,department_raw,tenure_years,review_date,review_text
E9998,Ops,2.0,2026-08-02,
E9999,Data,1.0,2026-08-02,n/a


In [0]:
good_reviews = saved_bronze.filter(
    F.length(F.trim(F.col("review_text"))) >= 15
)

print("Valid rows:", good_reviews.count())
display(good_reviews.limit(10))

Valid rows: 48


employee_id,department_raw,tenure_years,review_date,review_text
E1001,Eng,1.3,2026-06-28,Mandatory overtime is burning out the team.
E1002,Engineering,2.2,2026-07-20,The new data platform saves me hours every week.
E1003,Eng,6.7,2026-05-25,The workload has been unsustainable since the reorganization.
E1004,Engineering,5.4,2026-07-29,My manager supports the team and communicates clearly.
E1005,Engineering,1.1,2026-07-04,Technical requirements often change halfway through the project.
E1006,Eng,5.5,2026-05-23,I receive useful feedback during code reviews.
E1007,Sales,6.5,2026-05-05,The bonus formula is confusing and not transparent.
E1008,sales,2.2,2026-05-19,Pay has not kept up with the market.
E1009,sales,7.3,2026-08-02,The recognition program is genuinely motivating.
E1010,Sales,6.4,2026-06-20,Late-night messages have become part of the culture.


## 4. Silver Layer — Cleaning, Standardization & Deduplication

Review text is normalized by trimming whitespace, collapsing repeated spaces, and converting text to lowercase for duplicate detection. Duplicate employee/comment combinations are removed, department labels are standardized, and the cleaned schema is prepared for downstream analysis.

The result is a **40-row cleaned analytical dataset**.


In [0]:
from pyspark.sql import functions as F

saved_bronze = spark.table("bronze_employee_reviews")

good_reviews = saved_bronze.filter(
    F.length(F.trim(F.col("review_text"))) >= 15
)

normalized_reviews = good_reviews.withColumn(
    "review_text_clean",
    F.lower(
        F.trim(
            F.regexp_replace(
                F.col("review_text"),
                r"\s+",
                " "
            )
        )
    )
)

In [0]:
deduplicated_reviews = normalized_reviews.dropDuplicates(
    ["employee_id", "review_text_clean"]
)

print("Before deduplication:", normalized_reviews.count())
print("After deduplication:", deduplicated_reviews.count())

Before deduplication: 48
After deduplication: 40


In [0]:
department_mapping = {
    "Eng": "Engineering",
    "engineering": "Engineering",
    "sales": "Sales",
    "CustSuccess": "Customer Success",
    "customer success": "Customer Success",
    "Data Analytics": "Data",
    "data": "Data",
    "Ops": "Operations",
    "operations": "Operations",
    "Prod": "Product",
    "product": "Product",
    "hr": "HR",
    "Human Resources": "HR"
}

standardized_reviews = deduplicated_reviews.replace(
    department_mapping,
    subset=["department_raw"]
)

In [0]:
display(
    standardized_reviews
    .groupBy("department_raw")
    .count()
    .orderBy("department_raw")
)

department_raw,count
Customer Success,6
Data,6
Engineering,6
HR,4
Operations,6
Product,6
Sales,6


In [0]:
silver_reviews = (
    standardized_reviews
    .withColumnRenamed("department_raw", "department")
    .withColumn(
        "review_date",
        F.to_date("review_date")
    )
    .select(
        "employee_id",
        "department",
        "tenure_years",
        "review_date",
        "review_text",
        "review_text_clean"
    )
)

print("Silver rows:", silver_reviews.count())
silver_reviews.printSchema()

display(silver_reviews.limit(10))


Silver rows: 40
root
 |-- employee_id: string (nullable = true)
 |-- department: string (nullable = true)
 |-- tenure_years: double (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_text_clean: string (nullable = true)



employee_id,department,tenure_years,review_date,review_text,review_text_clean
E1001,Engineering,1.3,2026-06-28,Mandatory overtime is burning out the team.,mandatory overtime is burning out the team.
E1002,Engineering,2.2,2026-07-20,The new data platform saves me hours every week.,the new data platform saves me hours every week.
E1003,Engineering,6.7,2026-05-25,The workload has been unsustainable since the reorganization.,the workload has been unsustainable since the reorganization.
E1004,Engineering,5.4,2026-07-29,My manager supports the team and communicates clearly.,my manager supports the team and communicates clearly.
E1005,Engineering,1.1,2026-07-04,Technical requirements often change halfway through the project.,technical requirements often change halfway through the project.
E1006,Engineering,5.5,2026-05-23,I receive useful feedback during code reviews.,i receive useful feedback during code reviews.
E1007,Sales,6.5,2026-05-05,The bonus formula is confusing and not transparent.,the bonus formula is confusing and not transparent.
E1008,Sales,2.2,2026-05-19,Pay has not kept up with the market.,pay has not kept up with the market.
E1009,Sales,7.3,2026-08-02,The recognition program is genuinely motivating.,the recognition program is genuinely motivating.
E1010,Sales,6.4,2026-06-20,Late-night messages have become part of the culture.,late-night messages have become part of the culture.


## 5. AI-Assisted Sentiment Analysis

The cleaned Silver table is persisted in Databricks and enriched with `ai_analyze_sentiment()` to create an initial sentiment label for each employee comment.

A single-record test is run first, followed by the full Gold sentiment table.


In [0]:
%sql

SELECT ai_analyze_sentiment(
  'Mandatory overtime is burning out the team.'
) AS sentiment;

sentiment
negative


In [0]:
silver_reviews.write.mode("overwrite").saveAsTable(
    "workspace.default.silver_employee_reviews"
)

In [0]:
saved_silver = spark.table(
    "workspace.default.silver_employee_reviews"
)

print("Saved Silver rows:", saved_silver.count())

Saved Silver rows: 40


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_employee_sentiment AS

SELECT
  employee_id,
  department,
  tenure_years,
  review_date,
  review_text,
  ai_analyze_sentiment(review_text) AS sentiment

FROM workspace.default.silver_employee_reviews;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS total_rows
FROM workspace.default.gold_employee_sentiment;

total_rows
40


In [0]:
%sql

SELECT
  employee_id,
  department,
  review_text,
  sentiment
FROM workspace.default.gold_employee_sentiment
LIMIT 10;

employee_id,department,review_text,sentiment
E1013,Customer Success,Flexible hours help me balance work and family.,positive
E1030,Operations,Small errors often require several layers of approval to fix.,negative
E1005,Engineering,Technical requirements often change halfway through the project.,negative
E1028,Operations,The remote work policy helps me stay productive.,positive
E1010,Sales,Late-night messages have become part of the culture.,neutral
E1018,Customer Success,We do not have enough staff to manage the current workload.,negative
E1024,Product,Too many meetings leave little time for focused work.,negative
E1033,Data,My promotion salary increase was fair.,positive
E1023,Product,Cross-functional collaboration has improved this year.,positive
E1031,Data,Senior analysts provide excellent mentorship.,positive


## 6. Workforce Driver Classification

A second AI step assigns each review to a workforce-driver taxonomy. The taxonomy covers:

**Workload, Management, Compensation, Growth, WorkLifeBalance, Recognition, Tools, Process, Collaboration, Onboarding, and Other.**

The classification prompt explicitly defines each category and instructs the model to return exactly one category name.


In [0]:
%sql

SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  CONCAT(
    'Classify this employee review into exactly one category. ',
    'Use these definitions: ',
    'Workload = excessive tasks, meetings, staffing shortages, or insufficient time during work. ',
    'Management = manager or leadership behavior and communication. ',
    'Compensation = salary, bonus, or benefits. ',
    'Growth = promotion, learning, feedback, or career development. ',
    'WorkLifeBalance = after-hours work, weekends, flexibility, remote work, family or personal time. ',
    'Recognition = appreciation or credit for work. ',
    'Tools = systems, software, data access, or manual processes. ',
    'Other = none of the above. ',
    'Return only the category name. ',
    'Review: Too many meetings leave little time for focused work.'
  )
) AS driver;

driver
Workload


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_employee_signals AS

SELECT
  s.*,

  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Classify this employee review into exactly one category. ',
      'Use these definitions: ',
      'Workload = excessive tasks, meetings, staffing shortages, or insufficient time during work. ',
      'Management = manager or leadership behavior and communication. ',
      'Compensation = salary, bonus, or benefits. ',
      'Growth = promotion, learning, feedback, or career development. ',
      'WorkLifeBalance = after-hours work, weekends, flexibility, remote work, family or personal time. ',
      'Recognition = appreciation or credit for work. ',
      'Tools = systems, software, data access, or manual technical work. ',
      'Process = approvals, bureaucracy, inefficient procedures, or unclear workflows. ',
      'Collaboration = teamwork, cross-functional cooperation, or team support. ',
      'Onboarding = new-hire training, orientation, or onboarding documentation. ',
      'Other = none of the above. ',
      'Return only the category name. ',
      'Review: ',
      s.review_text
    )
  ) AS driver

FROM workspace.default.gold_employee_sentiment AS s;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
  review_text,
  sentiment,
  driver
FROM workspace.default.gold_employee_signals
LIMIT 10;

review_text,sentiment,driver
Flexible hours help me balance work and family.,positive,WorkLifeBalance
Small errors often require several layers of approval to fix.,negative,Process
Technical requirements often change halfway through the project.,negative,Process
The remote work policy helps me stay productive.,positive,WorkLifeBalance
Late-night messages have become part of the culture.,neutral,WorkLifeBalance
We do not have enough staff to manage the current workload.,negative,Workload
Processes are documented clearly and consistently.,positive,Process
Requirements frequently change during projects.,negative,Process
Mandatory overtime is burning out the team.,negative,WorkLifeBalance
The workload has been unsustainable since the reorganization.,negative,Workload


## 7. Human Review & Model Overrides

AI output is treated as a first-pass analytical label rather than unquestioned ground truth.

Reviews with **neutral sentiment** or an **Other** driver are surfaced for manual review. Three ambiguous comments are reviewed explicitly, and the workflow preserves:

- the original AI sentiment
- the final reviewed sentiment
- whether a human override occurred
- a short review note explaining the judgment

This keeps the analytical decision traceable.


In [0]:
%sql

SELECT
  review_text,
  sentiment,
  driver
FROM workspace.default.gold_employee_signals
WHERE sentiment = 'neutral'
   OR driver = 'Other'
ORDER BY sentiment, driver;

review_text,sentiment,driver
The team environment is fine but not especially engaging.,neutral,Collaboration
Late-night messages have become part of the culture.,neutral,WorkLifeBalance
Meetings take up most of the day and analysis happens at night.,neutral,Workload


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_employee_signals_reviewed AS

WITH reviewed AS (
  SELECT
    *,

    CASE
      WHEN review_text =
        'Late-night messages have become part of the culture.'
        THEN 'negative'

      WHEN review_text =
        'Meetings take up most of the day and analysis happens at night.'
        THEN 'negative'

      WHEN review_text =
        'The team environment is fine but not especially engaging.'
        THEN 'neutral'

      ELSE sentiment
    END AS reviewed_sentiment,

    CASE
      WHEN review_text =
        'Late-night messages have become part of the culture.'
        THEN 'Implicit after-hours work expectation.'

      WHEN review_text =
        'Meetings take up most of the day and analysis happens at night.'
        THEN 'Work is pushed into personal time.'

      WHEN review_text =
        'The team environment is fine but not especially engaging.'
        THEN 'Mild criticism, but not clearly negative.'

      ELSE NULL
    END AS review_note

  FROM workspace.default.gold_employee_signals
)

SELECT
  *,
  CASE
    WHEN reviewed_sentiment <> sentiment THEN true
    ELSE false
  END AS human_override

FROM reviewed;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
  review_text,
  sentiment AS ai_sentiment,
  reviewed_sentiment AS final_sentiment,
  human_override,
  review_note
FROM workspace.default.gold_employee_signals_reviewed
WHERE review_note IS NOT NULL;

review_text,ai_sentiment,final_sentiment,human_override,review_note
Late-night messages have become part of the culture.,neutral,negative,true,Implicit after-hours work expectation.
The team environment is fine but not especially engaging.,neutral,neutral,false,"Mild criticism, but not clearly negative."
Meetings take up most of the day and analysis happens at night.,neutral,negative,true,Work is pushed into personal time.


## 8. Management-Level Signal Analysis

The reviewed Gold table is summarized across several views:

- overall sentiment distribution
- review volume and negative rate by workforce driver
- department-level negative rates
- top negative drivers by department
- department risk summaries

These outputs are intended to surface **directional signals**, not to establish causal or organization-wide conclusions.


In [0]:
%sql

SELECT
  reviewed_sentiment,
  COUNT(*) AS review_count
FROM workspace.default.gold_employee_signals_reviewed
GROUP BY reviewed_sentiment
ORDER BY review_count DESC;

reviewed_sentiment,review_count
negative,22
positive,17
neutral,1


In [0]:
%sql

SELECT
  driver,
  COUNT(*) AS total_reviews,

  SUM(
    CASE
      WHEN reviewed_sentiment = 'negative' THEN 1
      ELSE 0
    END
  ) AS negative_reviews

FROM workspace.default.gold_employee_signals_reviewed

GROUP BY driver

ORDER BY negative_reviews DESC, total_reviews DESC;

driver,total_reviews,negative_reviews
Workload,5,5
Growth,8,3
Tools,4,3
Process,4,3
Management,5,2
WorkLifeBalance,4,2
Compensation,3,2
Recognition,2,1
Onboarding,2,1
Collaboration,3,0


In [0]:
%sql

SELECT
  driver,
  COUNT(*) AS total_reviews,
  SUM(
    CASE
      WHEN reviewed_sentiment = 'negative' THEN 1
      ELSE 0
    END
  ) AS negative_reviews,
  ROUND(
    100.0 * SUM(
      CASE
        WHEN reviewed_sentiment = 'negative' THEN 1
        ELSE 0
      END
    ) / COUNT(*),
    1
  ) AS negative_rate_pct

FROM workspace.default.gold_employee_signals_reviewed

GROUP BY driver

ORDER BY negative_rate_pct DESC, total_reviews DESC;

driver,total_reviews,negative_reviews,negative_rate_pct
Workload,5,5,100.0
Process,4,3,75.0
Tools,4,3,75.0
Compensation,3,2,66.7
WorkLifeBalance,4,2,50.0
Recognition,2,1,50.0
Onboarding,2,1,50.0
Management,5,2,40.0
Growth,8,3,37.5
Collaboration,3,0,0.0


In [0]:
%sql

SELECT
  department,

  COUNT(*) AS total_reviews,

  SUM(
    CASE
      WHEN reviewed_sentiment = 'negative' THEN 1
      ELSE 0
    END
  ) AS negative_reviews,

  ROUND(
    100.0 * SUM(
      CASE
        WHEN reviewed_sentiment = 'negative' THEN 1
        ELSE 0
      END
    ) / COUNT(*),
    1
  ) AS negative_rate_pct

FROM workspace.default.gold_employee_signals_reviewed

GROUP BY department

ORDER BY negative_rate_pct DESC, total_reviews DESC;

department,total_reviews,negative_reviews,negative_rate_pct
Sales,6,4,66.7
Customer Success,6,4,66.7
Product,6,3,50.0
Engineering,6,3,50.0
Operations,6,3,50.0
Data,6,3,50.0
HR,4,2,50.0


In [0]:
%sql

WITH negative_driver_counts AS (
  SELECT
    department,
    driver,
    COUNT(*) AS negative_count,
    ROW_NUMBER() OVER (
      PARTITION BY department
      ORDER BY COUNT(*) DESC, driver
    ) AS rank_in_department

  FROM workspace.default.gold_employee_signals_reviewed

  WHERE reviewed_sentiment = 'negative'

  GROUP BY department, driver
)

SELECT
  department,
  driver AS top_negative_driver,
  negative_count

FROM negative_driver_counts

WHERE rank_in_department = 1

ORDER BY negative_count DESC, department;

department,top_negative_driver,negative_count
Sales,Compensation,2
Customer Success,Growth,1
Data,Process,1
Engineering,Process,1
HR,Management,1
Operations,Process,1
Product,Growth,1


In [0]:
%sql

WITH negative_driver_counts AS (
  SELECT
    department,
    driver,
    COUNT(*) AS negative_count

  FROM workspace.default.gold_employee_signals_reviewed

  WHERE reviewed_sentiment = 'negative'

  GROUP BY department, driver
),

ranked_drivers AS (
  SELECT
    *,
    DENSE_RANK() OVER (
      PARTITION BY department
      ORDER BY negative_count DESC
    ) AS rank_in_department

  FROM negative_driver_counts
)

SELECT
  department,
  driver AS top_negative_driver,
  negative_count

FROM ranked_drivers

WHERE rank_in_department = 1

ORDER BY department, top_negative_driver;

department,top_negative_driver,negative_count
Customer Success,Growth,1
Customer Success,Onboarding,1
Customer Success,Recognition,1
Customer Success,Workload,1
Data,Process,1
Data,Tools,1
Data,Workload,1
Engineering,Process,1
Engineering,WorkLifeBalance,1
Engineering,Workload,1


In [0]:
%sql

SELECT
  department,
  COUNT(*) AS total_reviews,

  SUM(
    CASE
      WHEN reviewed_sentiment = 'negative' THEN 1
      ELSE 0
    END
  ) AS negative_reviews,

  ROUND(
    100.0 * SUM(
      CASE
        WHEN reviewed_sentiment = 'negative' THEN 1
        ELSE 0
      END
    ) / COUNT(*),
    1
  ) AS negative_rate_pct,

  CONCAT_WS(
    ', ',
    SORT_ARRAY(
      COLLECT_SET(
        CASE
          WHEN reviewed_sentiment = 'negative' THEN driver
        END
      )
    )
  ) AS negative_drivers

FROM workspace.default.gold_employee_signals_reviewed

GROUP BY department

ORDER BY negative_rate_pct DESC, department;

department,total_reviews,negative_reviews,negative_rate_pct,negative_drivers
Customer Success,6,4,66.7,"Growth, Onboarding, Recognition, Workload"
Sales,6,4,66.7,"Compensation, Growth, WorkLifeBalance"
Data,6,3,50.0,"Process, Tools, Workload"
Engineering,6,3,50.0,"Process, WorkLifeBalance, Workload"
HR,4,2,50.0,"Management, Tools"
Operations,6,3,50.0,"Process, Tools, Workload"
Product,6,3,50.0,"Growth, Management, Workload"


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_department_risk_summary AS

SELECT
  department,
  COUNT(*) AS total_reviews,

  SUM(
    CASE
      WHEN reviewed_sentiment = 'negative' THEN 1
      ELSE 0
    END
  ) AS negative_reviews,

  ROUND(
    100.0 * SUM(
      CASE
        WHEN reviewed_sentiment = 'negative' THEN 1
        ELSE 0
      END
    ) / COUNT(*),
    1
  ) AS negative_rate_pct,

  CONCAT_WS(
    ', ',
    SORT_ARRAY(
      COLLECT_SET(
        CASE
          WHEN reviewed_sentiment = 'negative' THEN driver
        END
      )
    )
  ) AS negative_drivers

FROM workspace.default.gold_employee_signals_reviewed

GROUP BY department;

num_affected_rows,num_inserted_rows


## 9. AI-Generated Management Brief

The aggregated statistics are passed to an LLM with explicit guardrails:

- use only the supplied evidence
- summarize sentiment using counts
- identify signals using both volume and negative rate
- avoid inventing causes or unsupported recommendations
- state that the data is synthetic and the findings are directional

This demonstrates how generative AI can support reporting while keeping the evidence boundary explicit.


In [0]:
%sql

WITH sentiment_counts AS (
  SELECT
    reviewed_sentiment,
    COUNT(*) AS review_count
  FROM workspace.default.gold_employee_signals_reviewed
  GROUP BY reviewed_sentiment
),

sentiment_summary AS (
  SELECT
    CONCAT_WS(
      ', ',
      SORT_ARRAY(
        COLLECT_LIST(
          CONCAT(
            reviewed_sentiment,
            ': ',
            CAST(review_count AS STRING)
          )
        )
      )
    ) AS summary_text
  FROM sentiment_counts
),

driver_stats AS (
  SELECT
    driver,
    COUNT(*) AS total_reviews,

    SUM(
      CASE
        WHEN reviewed_sentiment = 'negative' THEN 1
        ELSE 0
      END
    ) AS negative_reviews,

    SUM(
      CASE
        WHEN reviewed_sentiment = 'positive' THEN 1
        ELSE 0
      END
    ) AS positive_reviews,

    ROUND(
      100.0 * SUM(
        CASE
          WHEN reviewed_sentiment = 'negative' THEN 1
          ELSE 0
        END
      ) / COUNT(*),
      1
    ) AS negative_rate_pct

  FROM workspace.default.gold_employee_signals_reviewed
  GROUP BY driver
),

driver_summary AS (
  SELECT
    CONCAT_WS(
      ' | ',
      SORT_ARRAY(
        COLLECT_LIST(
          CONCAT(
            driver,
            ': ',
            CAST(total_reviews AS STRING),
            ' total, ',
            CAST(negative_reviews AS STRING),
            ' negative, ',
            CAST(positive_reviews AS STRING),
            ' positive, ',
            CAST(negative_rate_pct AS STRING),
            '% negative'
          )
        )
      )
    ) AS summary_text
  FROM driver_stats
),

department_summary AS (
  SELECT
    CONCAT_WS(
      ' | ',
      SORT_ARRAY(
        COLLECT_LIST(
          CONCAT(
            department,
            ': ',
            CAST(total_reviews AS STRING),
            ' total, ',
            CAST(negative_reviews AS STRING),
            ' negative, ',
            CAST(negative_rate_pct AS STRING),
            '% negative; negative drivers: ',
            negative_drivers
          )
        )
      )
    ) AS summary_text
  FROM workspace.default.gold_department_risk_summary
)

SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',

  CONCAT(
    'Write a concise management brief based only on the evidence below. ',
    'The dataset contains 40 cleaned, synthetic employee reviews. ',
    'Write 4 short sentences in professional business English. ',
    'Sentence 1: summarize the overall sentiment distribution using counts. ',
    'Sentence 2: identify the strongest negative workforce signal using both volume and negative rate. ',
    'Sentence 3: identify one positive or lower-risk signal. ',
    'Sentence 4: state that the findings are directional because the data is synthetic and recommend validating the patterns with a larger real dataset. ',
    'Do not invent causes, make unsupported recommendations, or claim that the results represent a real workforce. ',

    'Overall sentiment: ',
    sentiment_summary.summary_text,

    '. Driver statistics: ',
    driver_summary.summary_text,

    '. Department statistics: ',
    department_summary.summary_text
  )
) AS management_brief

FROM sentiment_summary
CROSS JOIN driver_summary
CROSS JOIN department_summary;

management_brief
"The overall sentiment distribution shows 22 negative reviews, 1 neutral review, and 17 positive reviews, indicating a predominantly negative sentiment. The strongest negative workforce signal is Workload, with 5 total mentions and a 100% negative rate, suggesting a critical area of concern. A relatively positive signal is Collaboration, with 3 total mentions, 2 positive reviews, and a 0% negative rate, indicating a potential strength. These findings are directional and based on synthetic data, so it is recommended to validate these patterns with a larger, real dataset to confirm their accuracy and inform meaningful business decisions."


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_management_brief_reviewed AS

SELECT
  'The overall sentiment distribution includes 22 negative, 17 positive, and 1 neutral review, with negative feedback slightly more common in this synthetic sample. Workload shows the clearest negative pattern, with all 5 related comments classified as negative. Collaboration appears comparatively positive, with 2 positive comments and no negative comments among 3 total mentions. These findings are directional and based on synthetic data, so they should be validated using a larger real-world dataset before informing business decisions.'
  AS reviewed_management_brief;

num_affected_rows,num_inserted_rows


## 10. Final Reviewed Brief

The management-facing summary below is the final reviewed interpretation of the synthetic dataset.


The overall sentiment distribution includes 22 negative, 17 positive, and 1 neutral review, with negative feedback slightly more common in this synthetic sample. Workload shows the clearest negative pattern, with all 5 related comments classified as negative. Collaboration appears comparatively positive, with 2 positive comments and no negative comments among 3 total mentions. These findings are directional and based on synthetic data, so they should be validated using a larger real-world dataset before informing business decisions.

## Limitations

- The dataset is **synthetic** and contains only 40 cleaned records.
- Driver-level findings can be based on very small numbers of comments.
- AI-generated sentiment and driver labels can be ambiguous and require human review.
- The resulting patterns should be treated as **emerging signals**, not validated workforce conclusions.
- A production implementation would require stronger validation, governance, privacy, and access-control design.
